# ECM generation: aged cells

`run_ageing_sim` can fit an equivalent circuit model (ECM) to the aged cell at the end of a degradation campaign, with the final degradation applied to the physics. The fit is the same two stage fit `generate_ecm` runs for a fresh cell:

1. **Pulse fit.** The physics model runs a virtual pulse characterisation, a pulse followed by a rest at every point of the SoC, temperature and C-rate grid (60 s and 30 min by default). An RC network is fitted to each pulse and its relaxation, which gives R0 and the RC pairs.
2. **Constant current refit.** A pulse does not show the resistance a cell builds up over a long charge or discharge. The physics model also runs a constant current charge and discharge at every temperature and C-rate of the grid, and the slowest RC pair is refitted to match them.

Notebook *14a ECM Generation Beginning of Life* covers the fit, the parameter tables and the MATLAB export for a fresh cell in more detail. This notebook fits the aged ECM, exports it to MATLAB and compares it with the fresh one.

## 1. A coarser grid

A degradation campaign already takes a while, so the aged fit uses a coarser grid than notebook 14a: 10 SoC points instead of 16 (still denser at both ends), three temperatures and three C-rates. Every breakpoint is also a breakpoint of the 14a grid.

The fresh ECM is fitted on the same coarse grid, so the fresh and aged tables compare point for point.

In [1]:
from breathe_simulate import api_interface as api
from breathe_simulate.ecm import ecm_options

cell_name = "Molicel P45B"

options = ecm_options(
    n_rc=2,
    soc_grid=[0.0, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 1.0],
    temp_grid_degC=[0.0, 20.0, 40.0],
    charge_c_rates=[0.5, 1.0, 2.0],
    discharge_c_rates=[0.5, 1.0, 2.0],
)

fresh = api.generate_ecm(cell_name, ecm=options)
fresh

Submitted run_sim job 0f40e0bb-ad82-4337-9b4c-46df3a3df274


EcmResults(2RC, 10 SoC x 3 T x 3 dch / 3 chg rates)

## 2. An aged ECM from a degradation campaign

Pass `ecm=True` (or an `ecm_options(...)` block) to `run_ageing_sim` and the fit runs on the aged cell at the end of the campaign. Here it gets the same coarse grid as the fresh fit.

In [2]:
from breathe_simulate.ageing import AgeingCycler, RptCycler

CAP_AH = 4.5

ageing = AgeingCycler(selected_unit="C", cell_capacity=CAP_AH).cyclic(
    I_chg=1.0,
    I_dch=-1.0,
    I_cut=0.05,
    V_max=4.2,
    V_min=2.5,
    t_rest_s=300,
    t_max_cv_s=3600,
)
rpt = RptCycler(selected_unit="C", cell_capacity=CAP_AH).build(
    I_chg=1.0,
    I_cut=0.05,
    V_max=4.2,
    V_min=2.5,
    capacity_checks=[{"current": 1.0, "reference": True}],
    pulses=[{"soc": 0.5, "current": -1.0, "duration_s": 30, "reference": True}],
    t_equilibration_s=1800,
)

result = api.run_ageing_sim(
    cell_name,
    ageing,
    rpt_cycler=rpt,
    max_cycles=200,
    rpt_every_n_cycles=50,
    ecm=options,
)

aged = result.ecm
aged

Running ageing campaign on 'Molicel P45B' (up to 200 cycles)...
Submitted run_ageing_sim job 83654419-abf7-47f6-bd25-5102cf32beb8
Campaign running... elapsed 00:04:30
Campaign finished: max_cycles at cycle 200 (wall clock 00:04:39)


EcmResults(2RC, 10 SoC x 3 T x 3 dch / 3 chg rates, aged (SoH 92.9 % at cycle 200))

`degradation_state` records the aged state the fit ran at.

In [3]:
aged.degradation_state

{'lli': 0.06782495649212339,
 'lam_ne': 0.03249448782836761,
 'lam_pe': 0.0,
 'soh_pct': 92.9340198045519,
 'at_cycle': 200}

## 3. Download the aged MATLAB parameter set

`save_mat` writes the aged ECM as a `.mat` file for the Breathe Simulate 1st, 2nd and 3rd order ECM blocks in Simulink, in the same layout as a fresh one (notebook 14a describes the fields). The file records the degradation state the tables were fitted at, in `meta.lli`, `meta.lampe` and `meta.lamne`, and the state of health in `ocv.soh`, so a parameter set for an aged cell can't be mistaken for a fresh one.

In [4]:
from IPython.display import FileLink
from scipy.io import loadmat

aged_mat_path = "Molicel_P45B_ecmParams_2RC_aged.mat"
aged.save_mat(aged_mat_path, cell_name=f"{cell_name} (cycle 200)")

aged_params = loadmat(aged_mat_path, simplify_cells=True)["ecmParams"]
print(
    f"LLI {aged_params['meta']['lli']:.4f}, "
    f"LAM PE {aged_params['meta']['lampe']:.4f}, "
    f"LAM NE {aged_params['meta']['lamne']:.4f}, "
    f"SoH {aged_params['ocv']['soh']:.3f}"
)
FileLink(aged_mat_path)

LLI 0.0678, LAM PE 0.0000, LAM NE 0.0325, SoH 0.929


D:\Git\Breathe_Design_Product\breathe_simulate\docs\examples\Molicel_P45B_ecmParams_2RC_aged.mat

In Jupyter, the link above downloads the file. In MATLAB, load it and point the ECM block at the struct:

```matlab
load("Molicel_P45B_ecmParams_2RC_aged.mat")  % creates ecmParams in the workspace
% Breathe Simulate 2nd order ECM block: set the parameter p to ecmParams
```

Use the block whose order matches `modelOrder`, here the 2nd order block for `n_rc=2`.

## 4. Fresh against aged

Both fits share one grid, so the tables compare point for point.

In [5]:
import numpy as np
import plotly.graph_objects as go

temp_index = int(np.argmin(np.abs(fresh.temp_grid_degC - 20.0)))  # 20 degC
rate_index = int(
    np.argmin(np.abs(fresh.discharge_c_rates - 1.0))
)  # 1C discharge pulses

fig = go.Figure()
for label, ecm in (("fresh", fresh), ("aged", aged)):
    fig.add_trace(
        go.Scatter(
            x=ecm.soc_grid,
            y=1000 * ecm.r0("discharge")[:, temp_index, rate_index],
            mode="lines+markers",
            name=label,
        )
    )
fig.update_layout(
    title="R0 vs SoC at 20 degC, 1C discharge pulses",
    xaxis_title="SoC",
    yaxis_title="R0 [mOhm]",
    legend_title="Cell state",
)
fig

A small helper rebuilds the pulse voltage from the fitted tables. Feeding the same current pulse through both fits shows what the resistance growth feels like at the terminals.

In [6]:
def pulse_voltage(
    ecm, soc_index, temp_index, rate_index, amps=None, t_pulse_s=60.0, t_rest_s=180.0
):
    """Modelled terminal voltage for one discharge pulse followed by a rest."""
    if amps is None:
        amps = ecm.discharge_c_rates[rate_index] * ecm.capacity
    v_ocv = float(
        np.interp(ecm.soc_grid[soc_index], ecm.ocv["soc"], ecm.ocv["mean_voltage_V"])
    )
    time_s = np.linspace(0.0, t_pulse_s + t_rest_s, 600)
    on = time_s <= t_pulse_s
    voltage = np.full_like(time_s, v_ocv)
    voltage[on] -= amps * ecm.r0("discharge")[soc_index, temp_index, rate_index]
    for k in range(1, ecm.n_rc + 1):
        r_k = ecm.r(k, "discharge")[soc_index, temp_index, rate_index]
        tau_k = r_k * ecm.c(k, "discharge")[soc_index, temp_index, rate_index]
        voltage[on] -= amps * r_k * (1.0 - np.exp(-time_s[on] / tau_k))
        u_end = amps * r_k * (1.0 - np.exp(-t_pulse_s / tau_k))
        voltage[~on] -= u_end * np.exp(-(time_s[~on] - t_pulse_s) / tau_k)
    return time_s, voltage, v_ocv

In [7]:
amps = fresh.discharge_c_rates[rate_index] * fresh.capacity

fig = go.Figure()
for label, ecm in (("fresh", fresh), ("aged", aged)):
    soc_index = int(np.argmin(np.abs(ecm.soc_grid - 0.5)))  # 0.5 SoC in each grid
    time_s, voltage, _ = pulse_voltage(
        ecm, soc_index, temp_index, rate_index, amps=amps
    )
    fig.add_trace(go.Scatter(x=time_s, y=voltage, mode="lines", name=label))
fig.update_layout(
    title="1C discharge pulse at 0.5 SoC and 20 degC, fresh against aged",
    xaxis_title="Time [s]",
    yaxis_title="Voltage [V]",
    legend_title="Cell state",
)
fig

The OCV curves explain the offset at the start of the pulses. Against SoC both curves span 0 to 1, so ageing shows up as a shift in voltage at a given SoC. Against charge content (SoC times each cell's own capacity) the aged curve also ends early, at the capacity the cell has lost.

In [8]:
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("OCV against SoC", "OCV against charge content"),
)
for label, ecm, colour in (("fresh", fresh, "#1f77b4"), ("aged", aged, "#d62728")):
    ocv = ecm.ocv
    name = f"{label} ({ecm.capacity:.2f} Ah)"
    fig.add_trace(
        go.Scatter(
            x=ocv["soc"],
            y=ocv["mean_voltage_V"],
            mode="lines",
            name=name,
            legendgroup=label,
            line={"color": colour},
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=ocv["soc"] * ecm.capacity,
            y=ocv["mean_voltage_V"],
            mode="lines",
            name=name,
            legendgroup=label,
            showlegend=False,
            line={"color": colour},
        ),
        row=1,
        col=2,
    )
fig.update_xaxes(title_text="SoC", row=1, col=1)
fig.update_xaxes(title_text="Charge content [Ah]", row=1, col=2)
fig.update_yaxes(title_text="Mean OCV [V]", row=1, col=1)
fig.update_layout(title="Mean OCV, fresh against aged", legend_title="Cell state")
fig

To compare the pulses from the same starting voltage instead of the same SoC, invert each cell's OCV curve for the SoC at which it rests at a target voltage, here the fresh OCV at 0.5 SoC. The aged SoC then falls between breakpoints, so the R and C tables are interpolated linearly in SoC. Each pulse follows a 5 s rest at 0 A.

In [9]:
def soc_at_ocv(ecm, voltage):
    """SoC at which the cell rests at ``voltage`` on its mean OCV curve."""
    ocv = ecm.ocv.sort_values("soc")
    volts = ocv["mean_voltage_V"].to_numpy()
    if np.any(np.diff(volts) < 0):
        raise ValueError("Mean OCV is not monotonic in SoC, so it cannot be inverted.")
    if not volts[0] <= voltage <= volts[-1]:
        raise ValueError(f"{voltage:.3f} V is outside the OCV range of this cell.")
    return float(np.interp(voltage, volts, ocv["soc"].to_numpy()))


def table_at(ecm, table, soc, temp_degC, c_rate):
    """A discharge table read at any SoC, linear between the SoC breakpoints."""
    i_temp = int(np.argmin(np.abs(ecm.temp_grid_degC - temp_degC)))
    i_rate = int(np.argmin(np.abs(ecm.discharge_c_rates - c_rate)))
    if not np.isclose(ecm.temp_grid_degC[i_temp], temp_degC):
        raise ValueError(f"{temp_degC:g} degC is not on this ECM's temperature grid.")
    if not np.isclose(ecm.discharge_c_rates[i_rate], c_rate):
        raise ValueError(f"{c_rate:g}C is not on this ECM's discharge C-rate grid.")
    column = table[:, i_temp, i_rate]
    finite = np.isfinite(column)
    return float(np.interp(soc, ecm.soc_grid[finite], column[finite]))


def pulse_from_soc(
    ecm, soc, temp_degC, c_rate, amps, t_pre_s=5.0, t_pulse_s=60.0, t_rest_s=180.0
):
    """Terminal voltage for a 0 A rest, one discharge pulse, then a rest."""
    v_ocv = float(np.interp(soc, ecm.ocv["soc"], ecm.ocv["mean_voltage_V"]))
    time_s = np.arange(0.0, t_pre_s + t_pulse_s + t_rest_s + 0.05, 0.1)
    t_on = time_s - t_pre_s  # time since the pulse started
    on = (t_on >= 0.0) & (t_on <= t_pulse_s)
    after = t_on > t_pulse_s
    voltage = np.full_like(time_s, v_ocv)
    voltage[on] -= amps * table_at(ecm, ecm.r0("discharge"), soc, temp_degC, c_rate)
    for k in range(1, ecm.n_rc + 1):
        r_k = table_at(ecm, ecm.r(k, "discharge"), soc, temp_degC, c_rate)
        tau_k = r_k * table_at(ecm, ecm.c(k, "discharge"), soc, temp_degC, c_rate)
        voltage[on] -= amps * r_k * (1.0 - np.exp(-t_on[on] / tau_k))
        u_end = amps * r_k * (1.0 - np.exp(-t_pulse_s / tau_k))
        voltage[after] -= u_end * np.exp(-(t_on[after] - t_pulse_s) / tau_k)
    return time_s, voltage


temp_degC, c_rate = 20.0, 1.0
v_start = float(np.interp(0.5, fresh.ocv["soc"], fresh.ocv["mean_voltage_V"]))
amps = c_rate * fresh.capacity  # the same current through both cells

fig = go.Figure()
for label, ecm in (("fresh", fresh), ("aged", aged)):
    soc = soc_at_ocv(ecm, v_start)
    time_s, voltage = pulse_from_soc(ecm, soc, temp_degC, c_rate, amps)
    fig.add_trace(
        go.Scatter(x=time_s, y=voltage, mode="lines", name=f"{label} (SoC {soc:.3f})")
    )
fig.update_layout(
    title=f"{c_rate:g}C discharge pulse from {v_start:.3f} V at {temp_degC:g} degC, "
    "fresh against aged",
    xaxis_title="Time [s]",
    yaxis_title="Voltage [V]",
    legend_title="Cell state",
)
fig

The long tables take the comparison further, for example the relative R0 growth at every shared operating point.

In [10]:
fresh_table = fresh.to_dataframe().set_index(
    ["soc", "temperature_degC", "c_rate", "direction"]
)
aged_table = aged.to_dataframe().set_index(
    ["soc", "temperature_degC", "c_rate", "direction"]
)

growth = (aged_table["R0_Ohm"] / fresh_table["R0_Ohm"] - 1) * 100
growth_table = growth.dropna().rename("R0 growth [%]").reset_index()
growth_table.head(12)

,soc,temperature_degC,c_rate,direction,R0 growth [%]
0,0.00,0.0,0.5,discharge,1.162638
1,0.00,0.0,1.0,discharge,0.917527
2,0.00,0.0,2.0,discharge,1.142891
3,0.00,20.0,0.5,discharge,1.031320
4,0.00,20.0,1.0,discharge,0.700729
5,0.00,20.0,2.0,discharge,0.711627
6,0.00,40.0,0.5,discharge,0.983690
7,0.00,40.0,1.0,discharge,0.678651
8,0.00,40.0,2.0,discharge,0.609683
9,0.05,0.0,0.5,discharge,1.162638


The same growth as a chart, grouped by temperature.

In [11]:
at_1c = growth_table[
    (growth_table["direction"] == "discharge") & (growth_table["c_rate"] == 1.0)
]

fig = go.Figure()
for temperature, group in at_1c.groupby("temperature_degC"):
    fig.add_trace(
        go.Bar(
            x=group["soc"].astype(str),
            y=group["R0 growth [%]"],
            name=f"{temperature:g} degC",
        )
    )
fig.update_layout(
    barmode="group",
    title="R0 growth after the campaign, 1C discharge pulses",
    xaxis_title="SoC",
    yaxis_title="R0 growth [%]",
    legend_title="Temperature",
)
fig